Part 0

The PragmatiCQA dataset

Key motivations and contributions:

1. When using LLM's as a researching tool, having a question answered literally isn't particulary helpful. A human answer typically provides more information and context besides what was asked by inferring pragmatical information from the asker. The dataset provides human generated conversational question-answer pairs where the pragmatic answers have additional information and context beyond what was asked, anticipating follow up questions and being generally cooperative.

2. Existing datasets exploring CQA evaluated answers based on factual correctness and not cooperative pragmatic reasoning. Additionally, the teachers weren't insentivised to answer cooperatively, only to complete their task - which is answer the question and not necessarily create a welcoming learning experience. The PragmatiCQA mitigates this in several ways. First, teachers are incentivised to keep the conversations longer, and the students rate the teacher's answers afterwards. The students are incentivised to learn from their teacher's answers. Both students and teachers chose the topic they're interested in and were paired accordingly.

3. The PragmatiCQA Dataset provides diverse topics (73) based on community fandoms spanning comics, movies, video games and music, so evaluations and training can be done better.

4. The metrics used for evaluation take into consideration the answer's accuracy, pragmatic reasoning, naturalness and faithfulness.


What makes this dataset challenging for NLP models?
What specific pragmatic phenomena does it target?



Examples:

1.

2.

3.

4.

5.

Part 1

The "Traditional" NLP Approach

In [2]:
from transformers import pipeline
question_answerer = pipeline("question-answering", model='distilbert-base-cased-distilled-squad')

Device set to use cpu


In [3]:
import dspy
from sentence_transformers import SentenceTransformer

# Load an extremely efficient local model for retrieval
model = SentenceTransformer("sentence-transformers/static-retrieval-mrl-en-v1", device="cpu")

# Create an embedder using the model's encode method
embedder = dspy.Embedder(model.encode)

# Traverse a directory and read html files - extract text from the html files
import os
from bs4 import BeautifulSoup
def read_html_files(directory):
    texts = []
    for filename in os.listdir(directory):
        if filename.endswith(".html"):
            with open(os.path.join(directory, filename), 'r', encoding='utf-8') as file:
                soup = BeautifulSoup(file, 'html.parser')
                texts.append(soup.get_text())
    return texts

In [4]:
# Parameters for the retriever
max_characters = 10000  # for truncating >99th percentile of documents
topk_docs_to_retrieve = 3  # number of documents to retrieve per search query


In [ ]:
import json
import os  

def read_data(filename, dataset_dir="../PragmatiCQA/data"):
    corpus = []
    with open(os.path.join(dataset_dir, filename), 'r') as f:
        for line in f:
            corpus.append(json.loads(line))
    return corpus

pcqa_test = read_data("test.jsonl")

In [6]:
lm = dspy.LM('xai/grok-3-mini', max_tokens=6000, temperature=0.1, top_p=0.9)
dspy.configure(lm=lm)
with open("grok_key.ini") as f:
    for line in f:
        if "XAI_API_KEY" in line and not line.strip().startswith("#"):
            key_value = line.strip().split("=")
            if len(key_value) == 2:
                os.environ["XAI_API_KEY"] = key_value[1].split()[0]

In [7]:
from dspy.evaluate import SemanticF1

# Instantiate the metric.
metric = SemanticF1(decompositional=True)

In [8]:
import pprint   
pprint.pprint(pcqa_test[0]['qas'][0])

{'a': 'The Legend of Zelda came out as early as 1986 for the Famicom in Japan, '
      'and was later released in the western world, including Europe and the '
      'US in 1987. Would you like to know about the story?',
 'a_meta': {'literal_obj': [{'endKey': '9cbccabd-66be-4a46-bd8b-f59a299c987d',
                             'startKey': '1f4f808a-8560-4894-b892-15fa3c33887a',
                             'text': 'FDS release February 21, 1986\n'},
                            {'endKey': '738bff65-b4f9-4660-bd18-79722ed67a40',
                             'startKey': 'a0d9d5c5-18bb-4be4-825e-fca2900db18e',
                             'text': 'The Legend of Zelda is the first '
                                     'installment of the Zelda series. '},
                            {'endKey': '738bff65-b4f9-4660-bd18-79722ed67a40',
                             'startKey': '738bff65-b4f9-4660-bd18-79722ed67a40',
                             'text': ' It centers its plot around a boy named 

In [14]:
topic_search = {}


for item in pcqa_test:
    community = item['community']
    if community not in topic_search:
        topic_search[community] = dspy.retrievers.Embeddings(embedder=embedder, corpus=read_html_files(f"../PragmatiCQA-sources/{community}"), k=topk_docs_to_retrieve)

In [52]:
literal_references = []
literal_predictions = []
pragmatic_references = []
pragmatic_predictions = []
rag_references = []
rag_predictions = []

for item in pcqa_test:
    topic = item['topic']
    gernre = item['genre']
    community = item['community']
    qas = item['qas'][0]
    question = qas['q']
    answer = qas['a']
    lit_spans = [l['text'] for l in qas['a_meta']['literal_obj']]
    lit_answer = ' '.join(lit_spans)
    prag_spans = [l['text'] for l in qas['a_meta']['pragmatic_obj']]
    prag_answer = ' '.join(prag_spans)
    rag_context = topic_search[community](question)

    print(question)
    
    result_literal = question_answerer(question=question,     context=lit_answer)
    literal_references.append(dspy.Example(question=question, response=answer, inputs={'context': lit_answer}))
    literal_predictions.append(dspy.Prediction(question = question, response = result_literal['answer']).with_inputs(lit_answer))
    print(f"Answer with literal context: '{result_literal['answer']}'.")
    
    result_pragmatic = question_answerer(question=question,     context=prag_answer)
    pragmatic_references.append(dspy.Example(question=question, response=answer, inputs={'context': prag_answer}))
    pragmatic_predictions.append(dspy.Prediction(question = question, response = result_pragmatic['answer']).with_inputs(prag_answer))
    print(f"Answer with pragmatic context: '{result_pragmatic['answer']}'.")
    
    result_rag = question_answerer(question=question,     context=" ".join(rag_context.passages))
    rag_references.append(dspy.Example(question=question, response=answer, inputs={'context': rag_context}))
    rag_predictions.append(dspy.Prediction(question = question, response = result_rag['answer']).with_inputs(" ".join(rag_context.passages)))
    print(f"Answer with rag context: '{result_rag['answer']}'.")


What year did the Legend of Zelda come out?
Answer with literal context: '1986'.
Answer with pragmatic context: '1986'.
Answer with rag context: '1986'.
What console is The Legend of Zelda designed for?
Answer with literal context: 'Famicom'.
Answer with pragmatic context: 'Nintendo Entertainment System'.
Answer with rag context: 'Game Boy Color'.
when did the legend of zelda last until?
Answer with literal context: 'first installment in the Zelda franchise'.
Answer with pragmatic context: 'April 23, 2019'.
Answer with rag context: 'June 19, 2011'.
When was the Legend of Zelda released?
Answer with literal context: 'August 22, 1987'.
Answer with pragmatic context: '1987'.
Answer with rag context: '1986'.
What kind of game is The Legend of Zelda?
Answer with literal context: 'Zelda'.
Answer with pragmatic context: 'roleplaying'.
Answer with rag context: 'multiplayer'.
What year was this game release?
Answer with literal context: '1986'.
Answer with pragmatic context: '1987'.
Answer with

In [ ]:

literal_examples = []
for pred, reference in zip(literal_predictions, literal_references):
    literal_examples.append(dspy.Example(example = reference, pred = pred).with_inputs("example" , "pred"))

pragmatic_examples = []
for pred, reference in zip(pragmatic_predictions, pragmatic_references):
    pragmatic_examples.append(dspy.Example(example = reference, pred = pred).with_inputs("example" , "pred"))

rag_examples = []
for pred, reference in zip(rag_predictions, rag_references ):
    rag_examples.append(dspy.Example(example = reference, pred = pred).with_inputs("example" , "pred"))

literal_score = metric.batch(literal_examples)
pragmatic_score = metric.batch(pragmatic_examples)
rag_score = metric.batch(rag_examples)

Processed 213 / 213 examples: 100%|██████████| 213/213 [00:09<00:00, 23.41it/s]  


In [56]:
#evaluate

print(f"Mean results with literal context: {sum(literal_score) / 213}")
print(f"Mean results with pragmatic context: {sum(pragmatic_score) / 213}")
print(f"Mean results with rag context: {sum(rag_score) / 213}")

Mean results with literal context: 0.4088523710545823
Mean results with pragmatic context: 0.3634055940982089
Mean results with rag context: 0.10846077147389875


Evaluation:

From the score number, we can clearly see the model had trouble providing a relevant answer using the retrieved context.
Scores overall weren't high either. 

Looking at some of the answers, all were short, some even had just a single token (like a year or a name).

The model attempted at giving a literal answer, whether true or not, without pragmatic inferrence and without providing additional context to the answer.

It also never provided potential follow-up questions, and didn't answer conversationally.

Also, this is just funny:

Who is Snoopy?

Answer with literal context: 'a dog'.

Answer with pragmatic context: 'loves root beer and pizza'.

Answer with rag context: 'Tarzan'.

In [58]:
cost = sum([x['cost'] for x in lm.history if x['cost'] is not None])  # in USD, as calculated by LiteLLM for certain providers
print(cost)

0.8963926000000042
